# 1장 1강: 인공신경망과 DNN — 실습문제 풀이

> **풀이 기준**: 프로젝트에 업로드된 `concrete_data.csv`(1030행 × 9열)로 코드를 실제 실행한 결과입니다.
>
> **가정한 내용**: 별도로 판단이 필요한 부분은 없었습니다. 문제에 주어진 가중치·편향·구조를 그대로 사용했습니다.

---

## 실습 준비

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("concrete_data.csv")
df.columns = df.columns.str.strip()

print("전체 데이터 크기:", df.shape)
print("열 이름:", df.columns.tolist())
print(df.head(3).to_string(index=False))

전체 데이터 크기: (1030, 9)
열 이름: ['cement', 'blast_furnace_slag', 'fly_ash', 'water', 'superplasticizer', 'coarse_aggregate', 'fine_aggregate', 'age', 'concrete_compressive_strength']
 cement  blast_furnace_slag  fly_ash  water  superplasticizer  coarse_aggregate  fine_aggregate  age  concrete_compressive_strength
  540.0                 0.0      0.0  162.0               2.5            1040.0           676.0   28                          79.99
  540.0                 0.0      0.0  162.0               2.5            1055.0           676.0   28                          61.89
  332.5               142.5      0.0  228.0               0.0             932.0           594.0  270                          40.27


---

## 실습 1. 콘크리트 데이터와 신경망의 입력·출력 연결하기

### 문제 1-1. 입력 특성 8개와 실제 정답 1개 구분하기

In [4]:
feature_columns = [
    "cement", "blast_furnace_slag", "fly_ash", "water",
    "superplasticizer", "coarse_aggregate", "fine_aggregate", "age"
]
target_column = "concrete_compressive_strength"

X = df[feature_columns].to_numpy()
y = df[target_column].to_numpy()

print("X.shape:", X.shape)
print("y.shape:", y.shape)
print("첫 번째 샘플 입력 X[0]:", X[0])
print("첫 번째 샘플 정답 y[0]:", y[0])
print("입력층 뉴런 수:", X.shape[1], "/ 출력층 뉴런 수:", 1)

X.shape: (1030, 8)
y.shape: (1030,)
첫 번째 샘플 입력 X[0]: [ 540.     0.     0.   162.     2.5 1040.   676.    28. ]
첫 번째 샘플 정답 y[0]: 79.99
입력층 뉴런 수: 8 / 출력층 뉴런 수: 1


#### 답변

**수식 및 계산 과정 / 구조 설명**

- `X = df[feature_columns].to_numpy()` → 재료·재령 8개 열을 1030행 × 8열 배열로 만들었습니다.
- `y = df[target_column].to_numpy()` → 압축강도 1개 열을 길이 1030의 1차원 배열로 만들었습니다.
- 신경망 구조: **입력층 8개(특성 8개) → … → 출력층 1개(압축강도 한 값)**

**Q1. `X.shape`와 `y.shape`는 각각 무엇이며, 각 숫자는 무엇을 뜻하나요?**

- `X.shape = (1030, 8)`: 샘플(행)이 1030개이고, 샘플 하나당 입력 특성이 8개라는 뜻입니다.
- `y.shape = (1030,)`: 샘플 1030개 각각에 실제 정답(압축강도)이 하나씩 있는 1차원 배열입니다. 쉼표 뒤가 비어 있는 것은 열 개념이 없는 1차원이라는 뜻입니다.

**Q2. 첫 샘플의 시멘트량, 물의 양, 재령, 실제 압축강도는 각각 얼마인가요?**

- 시멘트량 **540**, 물의 양 **162**, 재령 **28일**, 실제 압축강도 **79.99**입니다.
- `X[0]`의 위치 번호로는 시멘트 `[0]`, 물 `[3]`, 재령 `[7]`에 해당합니다.

**Q3. 입력층은 왜 8개이고 출력층은 왜 1개인가요? 샘플이 1030개이면 입력층도 1030개인가요?**

- 입력층의 크기는 **샘플 하나를 설명하는 특성의 개수**로 정해지므로 8개입니다.
- 출력층은 압축강도라는 **연속적인 수치 하나**를 내야 하므로 뉴런 1개입니다.
- 샘플 수 1030은 입력층 크기와 무관합니다. 같은 신경망에 1030개 샘플을 한 개씩(또는 묶어서) 넣을 뿐이며, 입력층은 여전히 8개입니다.

**Q4. 실제 압축강도 열을 `X`에 넣지 않는 이유는 무엇인가요? 지금 코드에서 압축강도를 예측했나요?**

- 압축강도는 모델이 맞혀야 할 정답입니다. 입력에 넣으면 정답을 미리 알려주는 셈이 되어 의미 있는 예측이 되지 않습니다(정답 누출).
- 지금 코드는 입력과 정답을 **나누어 정리한 것**뿐이며, 예측이나 학습은 하지 않았습니다.

---

## 실습 2. 뉴런의 계산과 여러 층의 구조 이해하기

### 문제 2-1. 가중치와 편향을 사용한 한 뉴런의 계산

In [5]:
x_small = df[["cement", "water"]].to_numpy()[0]
print("x_small:", x_small)

w = np.array([0.1, -0.2])
b = 5.0

part1 = w[0] * x_small[0]
part2 = w[1] * x_small[1]
print("part1 = w1 * x1 =", part1)
print("part2 = w2 * x2 =", part2)

z = part1 + part2 + b
print("z =", z)

z_b0 = part1 + part2 + 0.0
print("z_b0 (편향 0) =", z_b0)

w_zero = np.array([0.0, 0.0])
z_w0 = w_zero[0] * x_small[0] + w_zero[1] * x_small[1] + b
print("z_w0 (가중치 모두 0) =", z_w0)

x_small: [540. 162.]
part1 = w1 * x1 = 54.0
part2 = w2 * x2 = -32.4
z = 26.6
z_b0 (편향 0) = 21.6
z_w0 (가중치 모두 0) = 5.0


#### 답변

**수식 및 계산 과정**

$$z = w_1x_1 + w_2x_2 + b = 0.1 \times 540 + (-0.2) \times 162 + 5$$

- w₁x₁ = 0.1 × 540 = **54**
- w₂x₂ = −0.2 × 162 = **−32.4**
- z = 54 + (−32.4) + 5 = **26.6**
- 편향 0: z_b0 = 54 − 32.4 + 0 = **21.6**
- 가중치 모두 0: z_w0 = 0 × 540 + 0 × 162 + 5 = **5**

**Q1. 두 곱셈 결과와 최종 z는 각각 얼마인가요?**

- part1 = 54, part2 = −32.4이고, z = 54 + (−32.4) + 5 = **26.6**입니다.

**Q2. 편향을 5에서 0으로 바꾸면 무엇이 얼마나 달라지나요?**

- 가중합 부분(54 − 32.4 = 21.6)은 그대로이고, 마지막에 더하는 값만 5에서 0으로 바뀌므로 z가 **정확히 5만큼 줄어** 21.6이 됩니다. 편향은 입력과 관계없이 결과 전체를 일정량 이동시키는 역할을 합니다.

**Q3. 모든 가중치가 0이면 z는 무엇으로 결정되나요? 편향도 0일 수 있나요?**

- 입력값에 상관없이 모든 곱이 0이 되므로 z는 **편향 b(=5)만으로 결정**됩니다.
- 편향도 0일 수 있으며, 그러면 z = 0입니다. 다만 이런 값은 입력 정보를 전혀 반영하지 못합니다.

**Q4. z를 첫 샘플의 실제 정답 79.99 또는 학습된 예측값이라고 부를 수 있나요?**

- 둘 다 아닙니다. z는 **임의로 정한 가중치·편향으로 계산한 가중합**입니다. 실제 정답(79.99)은 데이터에 기록된 측정값이고, 학습된 예측값은 데이터로 가중치를 조정한 뒤에 나오는 값입니다. 이번에는 학습을 하지 않았습니다.

**Q5. 이 코드가 고전적인 퍼셉트론의 전체 동작을 구현한 것인가요?**

- 아닙니다. 퍼셉트론은 **가중합 → 편향 → 기준에 따라 0 또는 1 출력**까지 수행합니다. 이번 코드는 마지막 출력 결정(계단 함수) 직전까지만 계산했습니다.

---

### 문제 2-2. 입력층·은닉층·출력층과 FFNN·MLP·DNN 구분하기

In [6]:
# 구조: 입력 8 → 은닉 4 → 은닉 2 → 출력 1
h1_weights = 8 * 4
h1_biases = 4
h2_weights = 4 * 2
h2_biases = 2
out_weights = 2 * 1
out_biases = 1

total_weights = h1_weights + h2_weights + out_weights
total_biases = h1_biases + h2_biases + out_biases
total_params = total_weights + total_biases

print("은닉층1  가중치:", h1_weights, " 편향:", h1_biases, " 합:", h1_weights + h1_biases)
print("은닉층2  가중치:", h2_weights, " 편향:", h2_biases, " 합:", h2_weights + h2_biases)
print("출력층   가중치:", out_weights, " 편향:", out_biases, " 합:", out_weights + out_biases)
print("전체 가중치:", total_weights)
print("전체 편향:", total_biases)
print("전체 파라미터:", total_params)

은닉층1  가중치: 32  편향: 4  합: 36
은닉층2  가중치: 8  편향: 2  합: 10
출력층   가중치: 2  편향: 1  합: 3
전체 가중치: 42
전체 편향: 7
전체 파라미터: 49


#### 답변

**구조 설명**

| 층 | 뉴런 수 | 역할 | 가중치 | 편향 |
|---|---:|---|---:|---:|
| 입력층 | 8 | 콘크리트 특성 8개를 받아 다음 층으로 전달 | – | – |
| 은닉층 1 | 4 | 입력 8개를 조합해 중간 값 4개를 만듦 | 8 × 4 = 32 | 4 |
| 은닉층 2 | 2 | 은닉층 1의 값 4개를 다시 조합해 중간 값 2개를 만듦 | 4 × 2 = 8 | 2 |
| 출력층 | 1 | 중간 값 2개로 압축강도 한 값을 만듦 | 2 × 1 = 2 | 1 |
| **합계** | | | **42** | **7** |

전체 파라미터 = 42 + 7 = **49개**

**Q1. 각 층의 뉴런 수와 역할은 무엇인가요? 은닉층 4개·2개는 반드시 이 숫자여야 하나요?**

- 위 표와 같습니다. 입력층 8개와 출력층 1개는 데이터와 목표에 의해 정해지지만, **은닉층의 뉴런 수(4, 2)는 설계자가 정하는 값**입니다. 반드시 이 숫자일 필요는 없고 실습용 예시입니다.

**Q2. 이 구조를 FFNN, MLP, DNN이라고 설명할 수 있는 이유를 각각 적으세요.**

- **FFNN**: 정보가 8 → 4 → 2 → 1 한 방향으로만 흐르고 되돌아가는 순환 연결이 없습니다.
- **MLP**: 은닉층이 있고, 각 층의 뉴런이 앞 층의 모든 값과 연결된 완전연결 구조입니다.
- **DNN**: 은닉층이 2개로 여러 은닉층을 쌓은 구조입니다.
- 따라서 이 신경망은 세 이름에 모두 해당할 수 있습니다.

**Q3. 층별 가중치·편향 수와 전체 파라미터 수는 얼마인가요? 샘플 수 1030을 곱해야 하나요?**

- 은닉층1 32 + 4, 은닉층2 8 + 2, 출력층 2 + 1 → 전체 **49개**입니다.
- 샘플 수는 곱하지 않습니다. 모든 샘플이 **같은 가중치·편향을 공유**하기 때문에 파라미터 수는 구조로만 결정됩니다.

**Q4. 입력층을 제외하고 가중치·편향이 있는 계산층은 몇 개이며, 출력층도 파라미터를 가지나요?**

- 은닉층 1, 은닉층 2, 출력층의 **3개**입니다.
- 출력층도 가중치 2개와 편향 1개, 모두 **3개의 파라미터**를 가집니다.

---

## 과제. 실습에서 배운 계산과 구조 읽기 복습

### 문제 3-1. 세 번째 샘플의 계산과 작은 신경망 구조 확인

In [7]:
task_x = df[["cement", "water"]].to_numpy()[2]
task_y = df["concrete_compressive_strength"].to_numpy()[2]
print("세 번째 샘플 입력 (시멘트, 물):", task_x)
print("세 번째 샘플 실제 정답:", task_y)

w = np.array([0.1, -0.2])
b = 5.0
task_part1 = w[0] * task_x[0]
task_part2 = w[1] * task_x[1]
task_z = task_part1 + task_part2 + b
task_z_b0 = task_part1 + task_part2
print("part1 =", round(task_part1, 4), " part2 =", round(task_part2, 4))
print("z =", round(task_z, 4), " z_b0 =", round(task_z_b0, 4))

# 2 → 2 → 2 → 1 구조
l1_w, l1_b = 2 * 2, 2
l2_w, l2_b = 2 * 2, 2
l3_w, l3_b = 2 * 1, 1
print("은닉층1:", l1_w, "+", l1_b, " 은닉층2:", l2_w, "+", l2_b, " 출력층:", l3_w, "+", l3_b)
print("전체 가중치:", l1_w + l2_w + l3_w, " 전체 편향:", l1_b + l2_b + l3_b,
      " 전체 파라미터:", l1_w + l2_w + l3_w + l1_b + l2_b + l3_b)

세 번째 샘플 입력 (시멘트, 물): [332.5 228. ]
세 번째 샘플 실제 정답: 40.27
part1 = 33.25  part2 = -45.6
z = -7.35  z_b0 = -12.35
은닉층1: 4 + 2  은닉층2: 4 + 2  출력층: 2 + 1
전체 가중치: 10  전체 편향: 5  전체 파라미터: 15


#### 답변

**수식 및 계산 과정**

- w₁x₁ = 0.1 × 332.5 = **33.25**
- w₂x₂ = −0.2 × 228 = **−45.6**
- z = 33.25 + (−45.6) + 5 = **−7.35**
- z_b0 = 33.25 − 45.6 = **−12.35**
- 2 → 2 → 2 → 1 구조: (2×2 + 2) + (2×2 + 2) + (2×1 + 1) = 6 + 6 + 3 = **15개**

**Q1. 세 번째 입력과 실제 정답은 무엇인가요? 왜 위치 번호 `[2]`를 사용하나요?**

- 입력은 시멘트량 332.5, 물의 양 228이고, 실제 정답은 40.27입니다.
- Python의 위치 번호는 0부터 시작하므로 첫 번째가 `[0]`, 두 번째가 `[1]`, 세 번째가 `[2]`입니다.

**Q2. 두 곱셈 결과, z, z_b0는 각각 얼마인가요?**

- part1 = 33.25, part2 = −45.6, z = **−7.35**, z_b0 = **−12.35**입니다.

**Q3. z가 실제 정답과 다르거나 음수이면 코드가 반드시 잘못된 것인가요?**

- 아닙니다. 가중치와 편향은 학습되지 않은 임의의 값이므로 z가 정답과 다른 것은 당연합니다. 물의 양에 곱하는 가중치가 음수이고 물의 양이 크기 때문에 음수가 나온 것이며, 계산 자체는 올바릅니다.

**Q4. `2 → 2 → 2 → 1`의 입력층·은닉층·출력층과 전체 파라미터 수를 설명하세요.**

- 입력층 2개(시멘트, 물) → 은닉층1 2개 → 은닉층2 2개 → 출력층 1개(압축강도)입니다.
- 가중치 4 + 4 + 2 = 10개, 편향 2 + 2 + 1 = 5개로 전체 **15개**입니다.

**Q5. 샘플이 달라질 때 바꾼 것과 유지한 것은 무엇인가요? 학습을 수행했나요?**

- 바꾼 것: 입력값(332.5, 228)과 비교용 실제 정답(40.27)입니다.
- 유지한 것: 가중치 [0.1, −0.2], 편향 5, 계산식입니다.
- 가중치를 데이터로 조정하지 않았으므로 **학습은 수행하지 않았습니다.**

---

## 실습 마무리

**1. 퍼셉트론 내부에서 가중치와 편향은 각각 어떻게 사용하나요?**

- 가중치는 각 입력에 곱해 그 입력이 결과에 미치는 영향의 크기와 방향을 정합니다. 편향은 가중합 뒤에 더해 결과 전체를 일정량 이동시킵니다.

**2. 입력층·은닉층·출력층의 역할을 각각 한 문장으로 설명하세요.**

- 입력층: 샘플의 특성 값을 받아 신경망에 전달합니다.
- 은닉층: 앞 층의 값을 가중치·편향으로 조합해 중간 값을 만듭니다.
- 출력층: 최종적으로 필요한 결과(이번에는 압축강도 한 값)를 만듭니다.

**3. 하나의 신경망이 FFNN이면서 MLP이고 DNN일 수 있나요?**

- 있습니다. 순환 없이 한 방향으로 흐르고(FFNN), 완전연결된 은닉층을 가지며(MLP), 은닉층이 여러 개라면(DNN) 세 이름에 모두 해당합니다. 실습 2-2의 8 → 4 → 2 → 1 구조가 그 예입니다.

**4. 같은 뉴런의 가중치와 편향을 샘플마다 새로 만들었나요?**

- 아닙니다. 첫 번째와 세 번째 샘플 모두 같은 가중치 [0.1, −0.2]와 편향 5를 사용했습니다. 샘플마다 바뀐 것은 입력값뿐입니다.

**5. 이번 실습에서 신경망 학습이나 완전한 다층 순전파를 수행했나요?**

- 둘 다 수행하지 않았습니다. 한 뉴런의 가중합 계산과 구조의 파라미터 수 세기만 했습니다. 활성화 함수를 거쳐 여러 층을 계산하는 순전파와, 데이터로 가중치를 조정하는 학습은 이후 강의 범위입니다.